In [1]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values, Json
import logging
import json
import html
import re

In [2]:
import pandas as pd
from pathlib import Path
import re

In [7]:
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'proquestarticles_nite'
sourcedata = Path(r"D:\development\2026_python\2026_thecall\code\metadataanalysis\combined_metadata.csv")

In [8]:
# Cell 1: Load CSV and query DB for storeids where source_file_x is null
df_csv = pd.read_csv(sourcedata)

# Normalise StoreId for reliable matching
df_csv['StoreId'] = df_csv['StoreId'].astype(str).str.strip()

conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

cur.execute(
    sql.SQL("SELECT storeid FROM {schema}.{table} WHERE source_file_x IS NULL")
    .format(
        schema=sql.Identifier(TABLE_SCHEMA),
        table=sql.Identifier(TABLE_NAME)
    )
)
null_storeids = [str(row[0]).strip() for row in cur.fetchall()]
print(f"Found {len(null_storeids)} rows with null source_file_x")

Found 7 rows with null source_file_x


In [9]:
# Cell 2: Build column mapping between CSV columns and DB columns
COLUMN_MAP = {
    'ArticleType':       'articletype',
    'Authors':           'authors',
    'copyright':         'copyright',
    'documentType':      'documenttype',   # ⚠️ double-check this DB column name
    'entryDate':         'entrydate',
    'language':          'language',
    'languageOfSummary': 'languageofsummary',
    'placeOfPublication':'placeofpublication',
    'pubdate':           'pubdate',
    'pubtitle':          'pubtitle',
    'year':              'year',
    'DocumentURL':       'documenturl',    # ⚠️ double-check this DB column name
    'startPage':         'startpage',
    'FindACopy':         'findacopy',
    'Database':          'database',
    'Title':             'article_title',
}

In [11]:
# Cell 3: Match CSV rows to null storeids and batch update the DB
df_csv_indexed = df_csv.set_index('StoreId')

updated = 0
not_found = []

def to_python_type(val):
    """Convert numpy/pandas types to native Python types for psycopg2."""
    if pd.isna(val):
        return None
    if isinstance(val, (int, float, str, bool)):
        return val
    return val.item()

for storeid in null_storeids:
    if storeid not in df_csv_indexed.index:
        not_found.append(storeid)
        continue

    csv_row = df_csv_indexed.loc[storeid]
    if isinstance(csv_row, pd.DataFrame):
        csv_row = csv_row.iloc[0]

    set_clause = ', '.join(
        f"{db_col} = %s" for db_col in COLUMN_MAP.values()
    )

    values = [
        to_python_type(csv_row[csv_col])
        for csv_col in COLUMN_MAP.keys()
    ]
    values.append(storeid)

    query = f"""
        UPDATE {TABLE_SCHEMA}.{TABLE_NAME}
        SET {set_clause}
        WHERE storeid = %s
        AND source_file_x IS NULL
    """

    cur.execute(query, values)
    updated += 1

conn.commit()
print(f"Updated:   {updated} rows")
print(f"Not found in CSV: {len(not_found)} storeids")
if not_found:
    print("Missing storeids:", not_found)

Updated:   7 rows
Not found in CSV: 0 storeids


In [12]:
# Cell 4: Clean up
cur.close()
conn.close()
print("Connection closed.")

Connection closed.
